In [12]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Add src to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sem_proj.data.datasets import BoasDataset
from sem_proj.data.preprocessing import PreprocessingConfig
from sem_proj.data.transforms import RandomTimeShift, RandomAmplitudeScale, RandomGaussianNoise
from sem_proj.data.splits import get_train_subjects

In [13]:
# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figure"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Load preprocessing config
preprocess_config = PreprocessingConfig.from_yaml(
    PROJECT_ROOT / "configs" / "preprocess" / "notch_bandpass_resample_znorm.yaml"
)

# Load dataset and get a sample
train_subjects = get_train_subjects()
dataset = BoasDataset(
    subjects=train_subjects[:1],  # Just first subject for demo
    mode="headband",
    preprocess_config=preprocess_config,
    use_cache=True,
    transform_hb=None,
)

# Get 50th sample
x_original, y = dataset[49]
print(f"Loaded sample shape: {x_original.shape}")
print(f"Sample label: {y}")

Loading sub-100...
  ✓ Loaded sub-100 (headband) from cache
Loaded sample shape: torch.Size([2, 3840])
Sample label: 2


In [14]:
# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Define augmentation
time_shift = RandomTimeShift(max_shift_ratio=0.10)

In [15]:
# Apply augmentation
x_time_shift = time_shift(x_original.clone())

In [18]:
def plot_augmentation(x_original, x_augmented, title, figname):
    """
    Create a side-by-side comparison plot of original vs augmented signal.
    Save as high-quality PDF.
    """
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(2, 1, figure=fig, hspace=0.4, wspace=0.3)
    
    # Time axis in seconds (assuming 128 Hz for headband after preprocessing)
    sr = 128  # sampling rate after preprocessing
    time = np.arange(x_original.shape[1]) / sr
    
    # Original signal
    ax0 = fig.add_subplot(gs[0])
    for ch in range(x_original.shape[0]):
        ax0.plot(time, x_original[ch].numpy() + ch * 2, linewidth=1.5, alpha=0.8)
    ax0.set_ylabel("Amplitude", fontsize=20)
    ax0.set_title("Original Signal", fontsize=14, fontweight="bold")
    ax0.tick_params(axis='both', labelsize=16)
    ax0.grid(True, alpha=0.3)
    
    # Augmented signal
    ax1 = fig.add_subplot(gs[1])
    for ch in range(x_augmented.shape[0]):
        ax1.plot(time, x_augmented[ch].numpy() + ch * 2, linewidth=1.5, alpha=0.8)
    ax1.set_xlabel("Time (s)", fontsize=20)
    ax1.set_ylabel("Amplitude", fontsize=20)
    ax1.set_title(f"Augmented Signal - {title}", fontsize=14, fontweight="bold")
    ax1.tick_params(axis='both', labelsize=16)
    ax1.grid(True, alpha=0.3)
    
    # fig.suptitle(f"Data Augmentation: {title}", fontsize=16, fontweight="bold", y=0.995)
    
    # Save as high-quality PDF
    pdf_path = FIGURE_DIR / figname
    fig.savefig(pdf_path, format="pdf", dpi=300, bbox_inches="tight")
    print(f"Saved: {pdf_path}")
    plt.close(fig)

In [19]:
# Visualize and save augmentation
plot_augmentation(x_original, x_time_shift, "Random Time Shift (±10%)", "augmentation_time_shift.pdf")

print("\nAugmentation visualization saved to reports/figures/")

Saved: c:\Users\leand\projects\sem-proj\reports\figure\augmentation_time_shift.pdf

Augmentation visualization saved to reports/figures/
